In [ ]:
<VSCode.Cell id="#VSC-91b5718f" language="markdown"># Run Exporter and Build

This notebook runs the exporter script, performs a build, commits and pushes changes, and inspects the output JSON file.</VSCode.Cell>

In [1]:
<VSCode.Cell id="#VSC-2f4de8ff" language="python">
import subprocess
import os
import sys

# Ensure working directory is the repository root
os.chdir('/workspaces/Foot')
print('CWD:', os.getcwd())
</VSCode.Cell>

SyntaxError: invalid syntax (375059461.py, line 1)

In [ ]:
## 运行导出脚本

使用 `subprocess.run` 调用 `python src/collect/jj_export.py`，捕获 stdout 和 stderr，并在笔记本中显示输出。


In [ ]:
# run exporter script
r = subprocess.run([sys.executable, 'src/collect/jj_export.py'], capture_output=True, text=True)
print('export returncode', r.returncode)
print('stdout:\n', r.stdout)
print('stderr:\n', r.stderr)


In [ ]:
# run build
r = subprocess.run([sys.executable, '-m', 'src.build'], capture_output=True, text=True)
print('build returncode', r.returncode)
print('stdout:\n', r.stdout)
print('stderr:\n', r.stderr)


In [ ]:
# git operations
for cmd in [['git','add','site/data/jczq.json','site/data/picks.json'], ['git','commit','-m','Update data'], ['git','push']]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print('cmd', cmd, 'rc', r.returncode)
    if r.stdout:
        print('stdout:', r.stdout)
    if r.stderr:
        print('stderr:', r.stderr)


In [ ]:
# read head of jczq.json
import pathlib
path=pathlib.Path('site/data/jczq.json')
if path.exists():
    for line in path.read_text(encoding='utf-8').splitlines()[:60]:
        print(line)
else:
    print('file missing')


In [ ]:
# perform the series of commands requested by the user
import subprocess, os, textwrap

# helper function

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout + result.stderr

os.chdir("/workspaces/Foot")

# 1. export
out_export = run("python -m src.collect.jj_export")

# 2. inspect json
json_content = ""
try:
    with open("site/data/jczq.json", "r") as f:
        json_content = f.read(500)
except FileNotFoundError:
    json_content = "<file not found>"

# 3. build
out_build_full = run("python -m src.build")
out_build = out_build_full[-500:]

# 4. git operations
out_git = run("git add -A && git commit -m 'switch to 500 jczq source' || true && git push")

print("=== jj_export output ===")
print(out_export)
print("\n=== jczq.json first 500 chars ===")
print(json_content)
print("\n=== build output (last 500 chars) ===")
print(out_build)
print("\n=== git operations output ===")
print(out_git)
